# Notebook 07 — Full-Well Inference

**Responsibilities (plan §12.8):**
- Load frozen model artifacts from NB06 (calibrated model, feature cols, clip bounds, train medians)
- Load full candidate universe from NB03 (all candidates across both wells, not just annotated crops)
- Load mega feature table from NB04 (features for all candidates)
- Apply preprocessing pipeline (clip → fill NaN with train medians)
- Run calibrated model to produce `prob_true_spot_calibrated` for every candidate
- Apply decision threshold (from NB06 manifest)
- Export `full_well_predictions.parquet` (§3.11 schema)
- Export per-well and per-region summary statistics
- Visual QA: overlay predictions on full well (tiled) and on each annotated crop
- Spatial density map: predicted spot density across the well
- Comparison to mentor baseline: how many spots does the model find vs mentor alone
- Timing report

**This notebook does NOT:**
- Re-run detectors, re-extract features, re-train models
- Touch annotations or supervision tables
- Modify any artifact from NB01–NB06

## How to use
1. Run NB06 to completion at least once
2. Set `CFG['model_run_id_override']` to the RUN_ID of the NB06 run you want to use,
   OR leave as None to use the most recent NB06 artifacts
3. Run all cells top to bottom
4. Outputs go to `tables/` and `artifacts/reports/nb07_inference/`

## Key design decisions
- **Frozen pipeline**: clip bounds, train medians, and feature list are loaded from NB06 artifacts — never recomputed
- **All candidates scored**: every union candidate in the full well gets a probability, not just supervised ones
- **Threshold from manifest**: the decision threshold is read from the NB06 manifest JSON — never re-optimized here
- **Tiled visualization**: full wells are too large to display at once; we tile into grid cells and render each

In [ ]:
# -------------------------------------------------------------------
# REPO DISCOVERY
# -------------------------------------------------------------------
from pathlib import Path

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / ".git").exists() or (p / "registries").exists():
            return p
    raise RuntimeError("Could not locate repo root.")

REPO_ROOT = find_repo_root()
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
# -------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------
CFG = {
    # ── Directory layout ───────────────────────────────────────────────────
    "tables_dir":        "tables",
    "artifact_dir":      "artifacts/reports/nb07_inference",
    "manifest_dir":      "artifacts/manifests",
    "models_dir":        "artifacts/models",

    # ── Model selection ────────────────────────────────────────────────────
    # None = autodiscover most recent NB06 artifacts
    # Set to a specific RUN_ID string to pin a model version, e.g.:
    # "nb06_20260317T211040Z_786ee10f"
    "model_run_id_override":   None,

    # Which model within the NB06 run to use for inference
    # None = use the primary model from the manifest
    "model_id_override":       None,

    # ── Candidate universe ──────────────────────────────────────────────────
    # None = autodiscover most recent NB03 candidate_union
    "candidate_union_path_override":    None,
    "mega_feature_table_path_override": None,

    # ── Wells to score ─────────────────────────────────────────────────────
    # None = score all wells present in candidate_union
    "wells_to_score":          None,

    # ── Inference ──────────────────────────────────────────────────────────
    # Batch size for predict_proba (candidates per batch, for memory management)
    "inference_batch_size":    10_000,

    # ── Visualization ──────────────────────────────────────────────────────
    "run_visual_qa":           True,
    "run_density_map":         True,
    "run_mentor_comparison":   True,

    # Tile size for full-well overlay (pixels)
    "tile_size_px":            1024,
    # Only render tiles that contain at least this many predicted positives
    "tile_min_predictions":    1,
    # Max tiles to render per well (cap for speed)
    "max_tiles_per_well":      20,

    # Display contrast stretch percentiles
    "display_lo_pct":          1,
    "display_hi_pct":          99,
    "circle_radius_px":        10,

    # Density map resolution (downsample factor relative to full well)
    "density_map_bin_px":      64,

    # ── Comparison ─────────────────────────────────────────────────────────
    # Mentor vote column to use for comparison baseline
    "mentor_vote_col":         "vote_consensus_v2",
    # Distance threshold to call a prediction a match to mentor (pixels)
    "mentor_match_radius_px":  8.0,

    # ── Persistence ────────────────────────────────────────────────────────
    "write_predictions":       True,
    "write_reports":           True,

    # ── Provenance ─────────────────────────────────────────────────────────
    "random_seed":             42,
}
CFG

In [ ]:
# -------------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------------
import gc
import hashlib
import json
import os
import re
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import tifffile
from scipy.ndimage import gaussian_filter
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", category=FutureWarning)
print("Imports OK")

In [ ]:
# -------------------------------------------------------------------
# HELPERS
# -------------------------------------------------------------------
def ts_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

def log(msg: str) -> None:
    print(f"[{ts_utc()}] {msg}", flush=True)

def ensure_dir(path) -> Path:
    p = Path(path); p.mkdir(parents=True, exist_ok=True); return p

def make_run_id(prefix="nb07") -> str:
    t = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    h = hashlib.sha1(f"{t}|{os.getpid()}|nb07".encode()).hexdigest()[:8]
    return f"{prefix}_{t}_{h}"

def sha1_text(x: str) -> str:
    return hashlib.sha1(x.encode()).hexdigest()

def safe_to_parquet(df: pd.DataFrame, path) -> Path:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
    except Exception as e:
        csv = path.with_suffix(".csv")
        log(f"[warn] parquet failed ({e}); writing CSV -> {csv.name}")
        df.to_csv(csv, index=False); return csv
    return path

def write_json(obj, path) -> Path:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    return path

def latest_matching_file(directory: Path, patterns: list):
    candidates = []
    if not directory.exists(): return None
    for pat in patterns:
        candidates.extend(directory.glob(pat))
    candidates = [p for p in candidates if p.is_file()]
    if not candidates: return None
    candidates.sort(key=lambda p: (p.stat().st_mtime, str(p)), reverse=True)
    return candidates[0]

def pct_clip_img(img: np.ndarray, lo: float = 1.0, hi: float = 99.0) -> np.ndarray:
    lo_v = np.percentile(img, lo)
    hi_v = np.percentile(img, hi)
    return np.clip((img - lo_v) / max(hi_v - lo_v, 1e-8), 0, 1)

RUN_ID       = make_run_id("nb07")
TABLES_DIR   = ensure_dir(REPO_ROOT / CFG["tables_dir"])
REPORT_DIR   = ensure_dir(REPO_ROOT / CFG["artifact_dir"])
MANIFEST_DIR = ensure_dir(REPO_ROOT / CFG["manifest_dir"])
MODELS_DIR   = ensure_dir(REPO_ROOT / CFG["models_dir"])
log(f"RUN_ID = {RUN_ID}")

In [ ]:
# -------------------------------------------------------------------
# LOAD NB06 ARTIFACTS
# Discovers: manifest, calibrated model, feature_cols, clip_bounds,
# train_medians. All preprocessing is frozen from NB06.
# -------------------------------------------------------------------
log("=" * 90)
log("LOADING NB06 ARTIFACTS")

# ── Discover manifest ──────────────────────────────────────────────
if CFG["model_run_id_override"]:
    rid = CFG["model_run_id_override"]
    manifest_path = MANIFEST_DIR / f"{rid}_model_manifest.json"
    assert manifest_path.exists(), f"Manifest not found: {manifest_path}"
else:
    manifest_path = latest_matching_file(MANIFEST_DIR, ["nb06_*_model_manifest.json"])
    assert manifest_path, "No NB06 manifest found. Run NB06 first."

log(f"Using manifest: {manifest_path.name}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

NB06_RUN_ID       = manifest["run_id"]
PRIMARY_MODEL_ID  = CFG["model_id_override"] or manifest["primary_model_id"]
DECISION_THRESHOLD = float(manifest["thresholds"][PRIMARY_MODEL_ID])
CAL_METHOD         = manifest["cal_methods"][PRIMARY_MODEL_ID]
MODEL_VERSION      = manifest.get("model_registry_version", NB06_RUN_ID)
THRESHOLD_POLICY   = manifest.get("threshold_policy", "max_f1")

log(f"  NB06 run_id:       {NB06_RUN_ID}")
log(f"  primary_model_id:  {PRIMARY_MODEL_ID}")
log(f"  decision_threshold:{DECISION_THRESHOLD:.6f}  (policy={THRESHOLD_POLICY}, selected on cal split)")
log(f"  calibration:       {CAL_METHOD}")

# ── Load model ────────────────────────────────────────────────────
model_cal_path = latest_matching_file(
    MODELS_DIR, [f"{NB06_RUN_ID}_{PRIMARY_MODEL_ID}_calibrated.pkl"]
)
if model_cal_path is None:
    # fallback: any calibrated model pkl from this run
    model_cal_path = latest_matching_file(
        MODELS_DIR, [f"{NB06_RUN_ID}_*_calibrated.pkl"]
    )
assert model_cal_path and model_cal_path.exists(), \
    f"Calibrated model pkl not found in {MODELS_DIR}. Did NB06 run with write_models=True?"

log(f"  Loading model: {model_cal_path.name}")
t0 = time.perf_counter()
MODEL = joblib.load(model_cal_path)
log(f"  Model loaded in {time.perf_counter()-t0:.2f}s: {type(MODEL).__name__}")

# ── Load preprocessing artifacts ──────────────────────────────────
feature_cols_path = latest_matching_file(
    MODELS_DIR, [f"{NB06_RUN_ID}_*_feature_cols.pkl"]
)
clip_bounds_path = latest_matching_file(
    MODELS_DIR, [f"{NB06_RUN_ID}_clip_bounds.pkl"]
)
medians_path = latest_matching_file(
    MODELS_DIR, [f"{NB06_RUN_ID}_train_medians.pkl"]
)

assert feature_cols_path and feature_cols_path.exists(), \
    "feature_cols.pkl not found — did NB06 run with write_models=True?"

FEATURE_COLS  = joblib.load(feature_cols_path)
CLIP_BOUNDS   = joblib.load(clip_bounds_path)   if clip_bounds_path and clip_bounds_path.exists() else {}
TRAIN_MEDIANS = joblib.load(medians_path)        if medians_path and medians_path.exists() else pd.Series(dtype=float)

log(f"  Feature cols: {len(FEATURE_COLS)}")
log(f"  Clip bounds:  {len(CLIP_BOUNDS)} features")
log(f"  Train medians:{len(TRAIN_MEDIANS)} features")

# Verify preprocessing artifacts are consistent
missing_medians = [c for c in FEATURE_COLS if c not in TRAIN_MEDIANS.index]
if missing_medians:
    log(f"  [warn] {len(missing_medians)} features missing from train_medians — will fill with 0")
    for c in missing_medians:
        TRAIN_MEDIANS[c] = 0.0

In [ ]:
# -------------------------------------------------------------------
# LOAD CANDIDATE UNIVERSE AND FEATURES
# -------------------------------------------------------------------
log("=" * 90)
log("LOADING CANDIDATE UNIVERSE AND FEATURES")

def _discover_or_override(override_key, patterns, label):
    if CFG[override_key]:
        p = Path(CFG[override_key])
        assert p.exists(), f"Override path not found: {p}"
        return p
    found = latest_matching_file(TABLES_DIR, patterns)
    assert found and found.exists(), \
        f"{label} not found in {TABLES_DIR}. Run NB03/NB04 first."
    return found

union_path   = _discover_or_override("candidate_union_path_override",
                   ["*candidate_union.parquet"], "candidate_union")
feature_path = _discover_or_override("mega_feature_table_path_override",
                   ["*mega_feature_table.parquet"], "mega_feature_table")

log(f"  candidate_union:   {union_path.name}")
log(f"  mega_feature_table:{feature_path.name}")

t0 = time.perf_counter()
union_df   = pd.read_parquet(union_path)
feature_df = pd.read_parquet(feature_path)
log(f"  loaded in {time.perf_counter()-t0:.2f}s")
log(f"  union_df:    {len(union_df):,} candidates  wells={sorted(union_df['well_id'].unique())}")
log(f"  feature_df:  {len(feature_df):,} rows × {len(feature_df.columns)} cols")

# Filter to requested wells
if CFG["wells_to_score"]:
    wells = CFG["wells_to_score"]
    union_df   = union_df[union_df["well_id"].isin(wells)].copy()
    feature_df = feature_df[feature_df["well_id"].isin(wells)].copy()
    log(f"  Filtered to wells {wells}: {len(union_df):,} candidates")

# Verify union_id alignment
union_ids_union   = set(union_df["union_id"].unique())
union_ids_feature = set(feature_df["union_id"].unique()) if "union_id" in feature_df.columns else set()
if union_ids_feature:
    only_in_union   = union_ids_union - union_ids_feature
    only_in_feature = union_ids_feature - union_ids_union
    if only_in_union:
        log(f"  [warn] {len(only_in_union)} union_ids in union but not in feature table")
    if only_in_feature:
        log(f"  [warn] {len(only_in_feature)} union_ids in feature table but not in union")

# Check all model features are present in feature table
missing_features = [c for c in FEATURE_COLS if c not in feature_df.columns]
if missing_features:
    log(f"  [warn] {len(missing_features)} model features missing from feature table:")
    for c in missing_features[:10]:
        log(f"    {c}")
    log(f"  These will be filled with train medians during inference.")

In [ ]:
# -------------------------------------------------------------------
# PREPROCESSING PIPELINE (frozen from NB06)
# Apply in exact order: clip → fill NaN with train medians
# -------------------------------------------------------------------
def preprocess_features(df: pd.DataFrame) -> np.ndarray:
    """
    Apply frozen NB06 preprocessing to a feature DataFrame.
    Returns float32 numpy array with shape (n_rows, n_features).
    """
    # 1. Select and add missing columns
    X = pd.DataFrame(index=df.index)
    for col in FEATURE_COLS:
        if col in df.columns:
            X[col] = pd.to_numeric(df[col], errors="coerce")
        else:
            X[col] = float(TRAIN_MEDIANS.get(col, 0.0))

    # 2. Apply quantile clip bounds (train-derived)
    for col in FEATURE_COLS:
        if col in CLIP_BOUNDS:
            lo, hi = CLIP_BOUNDS[col]
            if np.isfinite(lo) and np.isfinite(hi):
                X[col] = X[col].clip(lower=lo, upper=hi)

    # 3. Fill NaN with train medians
    for col in FEATURE_COLS:
        med = float(TRAIN_MEDIANS.get(col, 0.0))
        X[col] = X[col].fillna(med)

    return X.values.astype(np.float32)

log("Preprocessing pipeline defined (frozen from NB06).")

In [ ]:
# -------------------------------------------------------------------
# INFERENCE — score all candidates in batches
# -------------------------------------------------------------------
log("=" * 90)
log("RUNNING INFERENCE")

# Merge features onto union (union_id is the join key)
if "union_id" in feature_df.columns:
    inference_df = union_df[["union_id", "well_id", "crop_id",
                              "union_medoid_well_y_px", "union_medoid_well_x_px",
                              "union_n_members"]].copy()
    inference_df = inference_df.merge(
        feature_df[["union_id"] + [c for c in FEATURE_COLS if c in feature_df.columns]],
        on="union_id", how="left"
    )
else:
    # feature_df rows already aligned with union_df
    inference_df = pd.concat([union_df.reset_index(drop=True),
                               feature_df.reset_index(drop=True)], axis=1)

n_total   = len(inference_df)
batch_sz  = CFG["inference_batch_size"]
n_batches = math.ceil(n_total / batch_sz)
probs     = np.zeros(n_total, dtype=np.float32)

import math
log(f"  Scoring {n_total:,} candidates in {n_batches} batch(es) of {batch_sz:,} ...")
t0 = time.perf_counter()
for batch_idx in range(n_batches):
    start = batch_idx * batch_sz
    end   = min(start + batch_sz, n_total)
    batch = inference_df.iloc[start:end]
    X_batch = preprocess_features(batch)
    probs[start:end] = MODEL.predict_proba(X_batch)[:, 1]
    if (batch_idx + 1) % 5 == 0 or batch_idx == n_batches - 1:
        elapsed = time.perf_counter() - t0
        rate    = (end / elapsed) if elapsed > 0 else 0
        log(f"  Batch {batch_idx+1}/{n_batches}  scored {end:,}/{n_total:,}  ({rate:.0f} cand/s)")

total_elapsed = time.perf_counter() - t0
log(f"  Inference complete in {total_elapsed:.2f}s  ({n_total/total_elapsed:.0f} cand/s)")

inference_df["prob_true_spot_raw"]       = probs
inference_df["prob_true_spot_calibrated"]= probs  # already calibrated (model is CalibratedClassifierCV)
inference_df["predicted_label"]          = (probs >= DECISION_THRESHOLD).astype(int)

n_predicted = int(inference_df["predicted_label"].sum())
log(f"  Predicted positives: {n_predicted:,} / {n_total:,} "
    f"({100*n_predicted/n_total:.2f}%)  threshold={DECISION_THRESHOLD:.4f}")

# Per-well summary
log("\nPer-well prediction summary:")
for well_id, g in inference_df.groupby("well_id"):
    n_pos = int(g["predicted_label"].sum())
    n_tot = len(g)
    log(f"  {well_id}: {n_pos:,} predicted spots / {n_tot:,} candidates "
        f"({100*n_pos/n_tot:.2f}%)")

In [ ]:
# -------------------------------------------------------------------
# BUILD full_well_predictions.parquet (§3.11 schema)
# -------------------------------------------------------------------
log("=" * 90)
log("BUILDING full_well_predictions")

now_iso = datetime.now(timezone.utc).isoformat()

predictions = pd.DataFrame({
    "prediction_id":             [sha1_text(f"{uid}|{PRIMARY_MODEL_ID}|nb07|{RUN_ID}")
                                  for uid in inference_df["union_id"]],
    "union_id":                  inference_df["union_id"].values,
    "well_id":                   inference_df["well_id"].values,
    "crop_id":                   inference_df.get("crop_id", pd.Series([None]*len(inference_df))).values,
    "well_y_px":                 inference_df["union_medoid_well_y_px"].values,
    "well_x_px":                 inference_df["union_medoid_well_x_px"].values,
    "union_n_members":           inference_df["union_n_members"].values,
    "model_id":                  PRIMARY_MODEL_ID,
    "model_version":             MODEL_VERSION,
    "nb06_run_id":               NB06_RUN_ID,
    "nb07_run_id":               RUN_ID,
    "split_id":                  None,   # full-well inference has no split
    "prob_true_spot":             inference_df["prob_true_spot_raw"].values,
    "prob_true_spot_calibrated":  inference_df["prob_true_spot_calibrated"].values,
    "decision_threshold":         DECISION_THRESHOLD,
    "predicted_label":            inference_df["predicted_label"].values,
    "calibration_method":         CAL_METHOD,
    "threshold_policy":           THRESHOLD_POLICY,
    "created_at_utc":             now_iso,
})

assert predictions["prediction_id"].is_unique, "prediction_id must be unique"
log(f"  {len(predictions):,} prediction rows")
display(predictions.head())

# Quick sanity: probability distribution
log("\nProbability distribution (all candidates):")
display(predictions["prob_true_spot_calibrated"].describe().round(4).to_frame())

In [ ]:
# -------------------------------------------------------------------
# MENTOR COMPARISON  [TOGGLE: run_mentor_comparison]
# Compare model predictions to mentor baseline.
# -------------------------------------------------------------------
if CFG["run_mentor_comparison"]:
    log("=" * 90)
    log("MENTOR COMPARISON")

    mentor_col = CFG["mentor_vote_col"]
    if mentor_col in inference_df.columns:
        mentor_pos   = inference_df[mentor_col].fillna(0) > 0
        model_pos    = inference_df["predicted_label"] == 1

        both         = (mentor_pos & model_pos).sum()
        only_model   = (~mentor_pos & model_pos).sum()
        only_mentor  = (mentor_pos & ~model_pos).sum()
        neither      = (~mentor_pos & ~model_pos).sum()

        log(f"  Mentor column: {mentor_col}")
        log(f"  Mentor positives:    {int(mentor_pos.sum()):,}")
        log(f"  Model positives:     {int(model_pos.sum()):,}")
        log(f"  Both agree (spot):   {int(both):,}")
        log(f"  Model only:          {int(only_model):,}  (model finds, mentor misses)")
        log(f"  Mentor only:         {int(only_mentor):,}  (mentor finds, model rejects)")
        log(f"  Neither:             {int(neither):,}")

        inference_df["mentor_positive"] = mentor_pos.astype(int)
        inference_df["agreement"] = np.select(
            [both.values if hasattr(both, 'values') else
             np.array([(mentor_pos.iloc[i] and model_pos.iloc[i]) for i in range(len(inference_df))]),
             np.array([(not mentor_pos.iloc[i] and model_pos.iloc[i]) for i in range(len(inference_df))]),
             np.array([(mentor_pos.iloc[i] and not model_pos.iloc[i]) for i in range(len(inference_df))])],
            ["both", "model_only", "mentor_only"],
            default="neither"
        )

        # Vectorized version (cleaner)
        conditions = [
            mentor_pos & model_pos,
            ~mentor_pos & model_pos,
            mentor_pos & ~model_pos,
        ]
        choices = ["both", "model_only", "mentor_only"]
        inference_df["agreement"] = np.select(conditions, choices, default="neither")

        predictions["mentor_positive"] = inference_df["mentor_positive"].values
        predictions["agreement"]       = inference_df["agreement"].values

        # Per-well comparison
        log("\nPer-well mentor vs model:")
        for well_id, g in inference_df.groupby("well_id"):
            mp = int((g[mentor_col].fillna(0) > 0).sum())
            pp = int((g["predicted_label"] == 1).sum())
            ag = int(((g[mentor_col].fillna(0) > 0) & (g["predicted_label"] == 1)).sum())
            log(f"  {well_id}: mentor={mp:,}  model={pp:,}  agreement={ag:,} "
                f"({100*ag/max(mp,1):.1f}% of mentor)")

        # Probability distribution for mentor-only vs model-only
        log("\nProbability stats by agreement category:")
        display(
            inference_df.groupby("agreement")["prob_true_spot_calibrated"]
            .describe().round(4)
        )
    else:
        log(f"[warn] Mentor column '{mentor_col}' not found in feature table. "
            f"Available vote columns: {[c for c in inference_df.columns if c.startswith('vote_')]}")
else:
    log("Mentor comparison disabled.")

In [ ]:
# -------------------------------------------------------------------
# PROBABILITY DISTRIBUTION PLOTS
# -------------------------------------------------------------------
log("=" * 90)
log("PROBABILITY DISTRIBUTION PLOTS")

n_wells = predictions["well_id"].nunique()
fig, axes = plt.subplots(1, n_wells + 1, figsize=(6 * (n_wells + 1), 4))
if n_wells == 0:
    axes = [axes]
elif not hasattr(axes, '__len__'):
    axes = [axes]

# All wells combined
ax = axes[0]
probs_all = predictions["prob_true_spot_calibrated"].values
ax.hist(probs_all, bins=100, log=True, color="steelblue", alpha=0.7, edgecolor="none")
ax.axvline(DECISION_THRESHOLD, color="red", linestyle="--", lw=2,
           label=f"threshold={DECISION_THRESHOLD:.3f}")
ax.set_xlabel("Calibrated probability"); ax.set_ylabel("Count (log scale)")
ax.set_title("All wells — probability distribution")
ax.legend(fontsize=8)

# Per-well
for i, (well_id, g) in enumerate(predictions.groupby("well_id"), 1):
    if i >= len(axes): break
    ax = axes[i]
    ax.hist(g["prob_true_spot_calibrated"].values, bins=80, log=True,
            color="darkorange", alpha=0.7, edgecolor="none")
    ax.axvline(DECISION_THRESHOLD, color="red", linestyle="--", lw=2)
    n_pos = int((g["predicted_label"] == 1).sum())
    ax.set_title(f"Well {well_id}\n{n_pos} predicted spots")
    ax.set_xlabel("Calibrated probability")

fig.tight_layout()
if CFG["write_reports"]:
    p = REPORT_DIR / f"{RUN_ID}_probability_distributions.png"
    fig.savefig(p, dpi=150, bbox_inches="tight")
    log(f"Saved: {p.name}")
plt.show(); plt.close(fig)

In [ ]:
# -------------------------------------------------------------------
# SPATIAL DENSITY MAP  [TOGGLE: run_density_map]
# Bin predicted spots into a 2D grid, smooth with Gaussian,
# visualize as a heatmap. No image loading required.
# -------------------------------------------------------------------
if CFG["run_density_map"]:
    log("=" * 90)
    log("SPATIAL DENSITY MAP")

    for well_id, g in predictions.groupby("well_id"):
        pos = g[g["predicted_label"] == 1]
        if len(pos) == 0:
            log(f"  {well_id}: no predicted spots — skipping density map")
            continue

        y_coords = pos["well_y_px"].dropna().values
        x_coords = pos["well_x_px"].dropna().values
        if len(y_coords) == 0:
            log(f"  {well_id}: no coordinate data — skipping")
            continue

        # Bin into grid
        bin_px   = CFG["density_map_bin_px"]
        y_max    = int(y_coords.max()) + bin_px
        x_max    = int(x_coords.max()) + bin_px
        n_bins_y = math.ceil(y_max / bin_px)
        n_bins_x = math.ceil(x_max / bin_px)

        density = np.zeros((n_bins_y, n_bins_x), dtype=np.float32)
        y_bin   = np.clip((y_coords / bin_px).astype(int), 0, n_bins_y - 1)
        x_bin   = np.clip((x_coords / bin_px).astype(int), 0, n_bins_x - 1)
        np.add.at(density, (y_bin, x_bin), 1)

        # Smooth
        smoothed = gaussian_filter(density, sigma=2.0)

        fig, axes_dm = plt.subplots(1, 2, figsize=(16, 7))

        im0 = axes_dm[0].imshow(density, cmap="hot", aspect="auto", interpolation="nearest")
        plt.colorbar(im0, ax=axes_dm[0], label="Spots per bin")
        axes_dm[0].set_title(f"Well {well_id} — raw spot density\n"
                             f"({bin_px}px bins, {len(pos)} predicted spots)")
        axes_dm[0].set_xlabel(f"X (bins of {bin_px}px)")
        axes_dm[0].set_ylabel(f"Y (bins of {bin_px}px)")

        im1 = axes_dm[1].imshow(smoothed, cmap="hot", aspect="auto", interpolation="bilinear")
        plt.colorbar(im1, ax=axes_dm[1], label="Smoothed density")
        axes_dm[1].set_title(f"Well {well_id} — smoothed density (σ=2 bins)")
        axes_dm[1].set_xlabel(f"X (bins of {bin_px}px)")

        fig.tight_layout()
        if CFG["write_reports"]:
            p = REPORT_DIR / f"{RUN_ID}_density_map_{well_id}.png"
            fig.savefig(p, dpi=150, bbox_inches="tight")
            log(f"  Saved: {p.name}")
        plt.show(); plt.close(fig)
else:
    log("Density map disabled.")

In [ ]:
# -------------------------------------------------------------------
# TILED FULL-WELL VISUAL QA  [TOGGLE: run_visual_qa]
# Loads actual image channels, tiles the well, renders predictions.
# Only tiles with predicted positives are shown (up to max_tiles_per_well).
# -------------------------------------------------------------------
if CFG["run_visual_qa"]:
    log("=" * 90)
    log("TILED FULL-WELL VISUAL QA")

    # ── Find data root ─────────────────────────────────────────────
    _data_candidates = [
        REPO_ROOT / "data",
        Path.home() / "Desktop" / "bpa1",
        Path.home() / "Desktop" / "BPA1",
        Path.home() / "Desktop",
    ]
    DATA_ROOT = None
    for cand in _data_candidates:
        if cand.exists() and list(cand.rglob("*.tiff")):
            DATA_ROOT = cand; break
    if DATA_ROOT is None:
        log("[warn] No .tiff files found — skipping visual QA. Set DATA_ROOT manually.")
    else:
        log(f"DATA_ROOT: {DATA_ROOT}")

        # ── Build image file index ──────────────────────────────────
        _WL_RE   = re.compile(r"(\d{3})_nm", re.IGNORECASE)
        _WELL_RE = re.compile(r"(?<![A-Z0-9])([A-H](?:[1-9]|1[0-2]))(?![A-Z0-9])", re.IGNORECASE)
        WL_LABELS = {"405": "DAPI", "561": "Ch561", "638": "Ch638", "BF": "BF"}
        SHOW_WLS  = ["638", "561"]  # channels to display; 638 is usually the spot channel

        def parse_well_wl(fname):
            wm = _WELL_RE.search(fname)
            well = wm.group(1).upper() if wm else None
            if "BF" in fname.upper(): wl = "BF"
            else:
                wlm = _WL_RE.search(fname)
                wl = wlm.group(1) if wlm else None
            return well, wl

        img_rows = []
        for ext in ["*.tiff", "*.tif"]:
            for f in DATA_ROOT.rglob(ext):
                if f.name.startswith("._"): continue
                well, wl = parse_well_wl(f.name)
                if well and wl:
                    img_rows.append({"well_id": well, "wavelength": wl, "path": f})

        if not img_rows:
            log("[warn] No image files parsed — check filename pattern")
        else:
            img_index = pd.DataFrame(img_rows).drop_duplicates(subset=["well_id", "wavelength"])
            log(f"Image index: {len(img_index)} files  wells={sorted(img_index['well_id'].unique())}")

            def load_well_channel(well_id, wavelength):
                row = img_index[(img_index["well_id"]==well_id) &
                                (img_index["wavelength"]==wavelength)]
                if len(row) == 0: return None
                arr = np.asarray(tifffile.imread(str(row.iloc[0]["path"]))).squeeze()
                if arr.ndim == 3: arr = arr.max(axis=0)
                return arr.astype(np.float32)

            # ── Render tiled overlays ────────────────────────────────
            tile_px = CFG["tile_size_px"]
            R       = CFG["circle_radius_px"]

            for well_id in predictions["well_id"].unique():
                log(f"\nRendering well {well_id} ...")

                # Load primary display channel
                primary_wl = next((wl for wl in SHOW_WLS
                                   if any((img_index["well_id"]==well_id) &
                                          (img_index["wavelength"]==wl))), None)
                if primary_wl is None:
                    log(f"  [skip] No display channel for well {well_id}")
                    continue

                img = load_well_channel(well_id, primary_wl)
                if img is None:
                    log(f"  [skip] Failed to load {well_id} wl={primary_wl}")
                    continue

                H, W = img.shape
                log(f"  Image shape: {H}×{W}  channel={primary_wl}")

                # Get predictions for this well
                well_preds = predictions[
                    (predictions["well_id"] == well_id) &
                    (predictions["predicted_label"] == 1)
                ].copy()
                log(f"  {len(well_preds)} predicted spots")

                if len(well_preds) == 0:
                    log(f"  [skip] No predicted spots in well {well_id}")
                    continue

                # Build tile grid
                n_tiles_y = math.ceil(H / tile_px)
                n_tiles_x = math.ceil(W / tile_px)
                log(f"  Tile grid: {n_tiles_y}×{n_tiles_x} ({n_tiles_y*n_tiles_x} tiles)")

                # Find tiles with predicted spots
                well_preds = well_preds.dropna(subset=["well_y_px", "well_x_px"])
                well_preds["tile_y"] = (well_preds["well_y_px"] / tile_px).astype(int).clip(0, n_tiles_y-1)
                well_preds["tile_x"] = (well_preds["well_x_px"] / tile_px).astype(int).clip(0, n_tiles_x-1)

                tiles_with_spots = (
                    well_preds.groupby(["tile_y", "tile_x"]).size()
                    .reset_index(name="n_spots")
                    .query(f"n_spots >= {CFG['tile_min_predictions']}")
                    .sort_values("n_spots", ascending=False)
                    .head(CFG["max_tiles_per_well"])
                )
                log(f"  Rendering {len(tiles_with_spots)} tiles with spots "
                    f"(max={CFG['max_tiles_per_well']})")

                # Also load second channel if available
                second_wl   = next((wl for wl in SHOW_WLS if wl != primary_wl
                                    and any((img_index["well_id"]==well_id) &
                                            (img_index["wavelength"]==wl))), None)
                img2 = load_well_channel(well_id, second_wl) if second_wl else None
                n_panels = 2 if img2 is not None else 1

                for _, tile_row in tiles_with_spots.iterrows():
                    ty, tx = int(tile_row["tile_y"]), int(tile_row["tile_x"])
                    y0 = ty * tile_px; y1 = min(y0 + tile_px, H)
                    x0 = tx * tile_px; x1 = min(x0 + tile_px, W)

                    # Candidates in this tile
                    tile_preds = well_preds[
                        (well_preds["tile_y"] == ty) & (well_preds["tile_x"] == tx)
                    ]

                    fig, axes_tile = plt.subplots(1, n_panels,
                                                  figsize=(5.5 * n_panels, 5.5),
                                                  gridspec_kw={"wspace": 0.03})
                    if n_panels == 1:
                        axes_tile = [axes_tile]

                    panels = [(primary_wl, img[y0:y1, x0:x1])]
                    if img2 is not None:
                        panels.append((second_wl, img2[y0:y1, x0:x1]))

                    for ax, (wl, tile_img) in zip(axes_tile, panels):
                        ax.imshow(pct_clip_img(tile_img,
                                               CFG["display_lo_pct"],
                                               CFG["display_hi_pct"]),
                                  cmap="gray", origin="upper")
                        ax.set_title(WL_LABELS.get(wl, wl), fontsize=10)
                        ax.set_xticks([]); ax.set_yticks([])

                        # Draw predicted spots
                        for _, pred_row in tile_preds.iterrows():
                            cy = float(pred_row["well_y_px"]) - y0
                            cx = float(pred_row["well_x_px"]) - x0
                            prob = float(pred_row["prob_true_spot_calibrated"])

                            # Color by agreement with mentor
                            if "agreement" in pred_row:
                                color = {"both": "lime", "model_only": "red",
                                         "mentor_only": "orange"}.get(pred_row["agreement"], "lime")
                            else:
                                color = "lime"

                            ax.add_patch(patches.Circle(
                                (cx, cy), R, lw=1.5,
                                edgecolor=color, facecolor="none", zorder=5
                            ))
                            # Annotate probability
                            ax.text(cx, cy - R - 2, f"{prob:.2f}",
                                    color=color, fontsize=5, ha="center",
                                    va="bottom", zorder=6)

                    # Legend
                    from matplotlib.lines import Line2D
                    legend_els = [
                        patches.Patch(facecolor="none", edgecolor="lime",   lw=1.5, label="Model + Mentor"),
                        patches.Patch(facecolor="none", edgecolor="red",    lw=1.5, label="Model only"),
                        patches.Patch(facecolor="none", edgecolor="orange", lw=1.5, label="Mentor only"),
                    ] if "agreement" in predictions.columns else [
                        patches.Patch(facecolor="none", edgecolor="lime",   lw=1.5, label="Predicted spot"),
                    ]
                    axes_tile[-1].legend(handles=legend_els, loc="upper right",
                                         fontsize=7, framealpha=0.85)

                    n_sp = int(tile_row["n_spots"])
                    fig.suptitle(
                        f"Well {well_id}  |  Tile ({ty},{tx})  "
                        f"y:[{y0},{y1})  x:[{x0},{x1})\n"
                        f"{n_sp} predicted spots  threshold={DECISION_THRESHOLD:.3f}",
                        fontsize=9, y=1.01
                    )
                    plt.tight_layout()
                    if CFG["write_reports"]:
                        p = REPORT_DIR / f"{RUN_ID}_tile_{well_id}_t{ty}_{tx}.png"
                        plt.savefig(p, dpi=160, bbox_inches="tight")
                    plt.show(); plt.close(fig)

        log("Visual QA complete.")
else:
    log("Visual QA disabled (CFG['run_visual_qa']=False).")

In [ ]:
# -------------------------------------------------------------------
# SUMMARY STATISTICS REPORT
# -------------------------------------------------------------------
log("=" * 90)
log("SUMMARY STATISTICS")

summary_rows = []
for well_id, g in predictions.groupby("well_id"):
    n_cands   = len(g)
    n_pos     = int((g["predicted_label"] == 1).sum())
    prob_vals = g["prob_true_spot_calibrated"].values
    row = {
        "well_id":             well_id,
        "n_candidates":        n_cands,
        "n_predicted_spots":   n_pos,
        "spot_rate":           round(n_pos / max(n_cands, 1), 4),
        "prob_mean":           round(float(prob_vals.mean()), 4),
        "prob_median":         round(float(np.median(prob_vals)), 4),
        "prob_p95":            round(float(np.percentile(prob_vals, 95)), 4),
        "prob_max":            round(float(prob_vals.max()), 4),
        "decision_threshold":  DECISION_THRESHOLD,
        "model_id":            PRIMARY_MODEL_ID,
        "nb06_run_id":         NB06_RUN_ID,
    }
    if "agreement" in g.columns:
        row["n_both_model_mentor"]  = int((g["agreement"] == "both").sum())
        row["n_model_only"]         = int((g["agreement"] == "model_only").sum())
        row["n_mentor_only"]        = int((g["agreement"] == "mentor_only").sum())
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

# Probability score bands
log("\nScore band breakdown (all wells):")
bands = [(0.9, 1.0, "very_high"), (0.7, 0.9, "high"),
         (0.5, 0.7, "medium"), (DECISION_THRESHOLD, 0.5, "low_above_thresh"),
         (0.0, DECISION_THRESHOLD, "below_thresh")]
band_rows = []
for lo, hi, label in bands:
    n = int(((predictions["prob_true_spot_calibrated"] >= lo) &
             (predictions["prob_true_spot_calibrated"] < hi)).sum())
    band_rows.append({"band": label, "prob_range": f"[{lo:.3f},{hi:.3f})", "n": n})
display(pd.DataFrame(band_rows))

In [ ]:
# -------------------------------------------------------------------
# PERSIST ALL OUTPUTS
# -------------------------------------------------------------------
log("=" * 90)
log("PERSISTING OUTPUTS")

output_paths = {}

if CFG["write_predictions"]:
    p = safe_to_parquet(
        predictions,
        TABLES_DIR / f"{RUN_ID}_full_well_predictions.parquet"
    )
    output_paths["full_well_predictions"] = str(p)
    log(f"  full_well_predictions: {p.name}  ({len(predictions):,} rows)")

if CFG["write_reports"]:
    p = safe_to_parquet(summary_df, REPORT_DIR / f"{RUN_ID}_inference_summary.parquet")
    output_paths["inference_summary"] = str(p)
    log(f"  inference_summary: {p.name}")

# NB07 manifest
nb07_manifest = {
    "run_id":                RUN_ID,
    "notebook_name":         "07_full_well_inference.ipynb",
    "created_at_utc":        datetime.now(timezone.utc).isoformat(),
    "nb06_run_id":           NB06_RUN_ID,
    "model_id":              PRIMARY_MODEL_ID,
    "model_version":         MODEL_VERSION,
    "decision_threshold":    DECISION_THRESHOLD,
    "threshold_policy":      THRESHOLD_POLICY,
    "calibration_method":    CAL_METHOD,
    "n_features":            len(FEATURE_COLS),
    "n_candidates_scored":   len(predictions),
    "n_predicted_spots":     int((predictions["predicted_label"] == 1).sum()),
    "wells_scored":          sorted(predictions["well_id"].unique().tolist()),
    "timing_seconds":        total_elapsed,
    "inference_rate_cands_per_sec": round(len(predictions) / max(total_elapsed, 1e-6), 1),
    "inputs": {
        "candidate_union":    str(union_path),
        "mega_feature_table": str(feature_path),
        "model":              str(model_cal_path),
        "manifest":           str(manifest_path),
    },
    "outputs":               output_paths,
    "per_well_summary":       summary_df.to_dict(orient="records"),
}
manifest_out = write_json(nb07_manifest, MANIFEST_DIR / f"{RUN_ID}_inference_manifest.json")
log(f"  Manifest: {manifest_out.name}")

In [ ]:
# -------------------------------------------------------------------
# EXIT CHECKS
# -------------------------------------------------------------------
log("=" * 90)
log("EXIT CHECKS")

# Schema check (§3.11)
required_cols = {
    "prediction_id", "union_id", "model_id", "model_version",
    "split_id", "prob_true_spot", "prob_true_spot_calibrated",
    "decision_threshold", "predicted_label", "calibration_method",
}
missing = sorted(required_cols - set(predictions.columns))
assert not missing, f"predictions missing required cols: {missing}"
assert predictions["prediction_id"].is_unique, "prediction_id not unique"

# Probability bounds
assert predictions["prob_true_spot_calibrated"].between(0, 1).all(), \
    "Calibrated probabilities out of [0,1]"

# Predicted label consistency
expected_pos = (predictions["prob_true_spot_calibrated"] >= DECISION_THRESHOLD).sum()
actual_pos   = (predictions["predicted_label"] == 1).sum()
assert expected_pos == actual_pos, \
    f"Predicted label mismatch: {actual_pos} labeled vs {expected_pos} expected from threshold"

log("All exit checks passed ✓")

log(f"\n{'=' * 90}")
log(f"NOTEBOOK 07 COMPLETE")
log(f"  NB06 run:              {NB06_RUN_ID}")
log(f"  Model:                 {PRIMARY_MODEL_ID}")
log(f"  Decision threshold:    {DECISION_THRESHOLD:.6f}  (policy={THRESHOLD_POLICY})")
log(f"  Candidates scored:     {len(predictions):,}")
log(f"  Predicted spots:       {int((predictions['predicted_label']==1).sum()):,}")
for well_id, g in predictions.groupby('well_id'):
    n = int((g['predicted_label']==1).sum())
    log(f"    {well_id}: {n:,} spots")
log(f"  Inference time:        {total_elapsed:.2f}s")
log(f"  Output:                {output_paths.get('full_well_predictions', 'not written')}")
log(f"{'=' * 90}")